# SegmentEveryForam Workflow: Foraminiferal Segmentation and Morphometric Analysis

# Project Description

This project aims to extract size information from foram images. The process involved segmenting forams, interacting with the segmented object, and extracting morphometric features. 

**Adapted from Segmenteverygrain:** https://github.com/zsylvester/segmenteverygrain

### Import Libraries and load models (We only do this once)

In [1]:
from matplotlib import pyplot as plt
import pandas as pd
import keras
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import segmenteveryforam as sef
import segmenteveryforam.interactions as sfi
from tqdm import tqdm
from PIL import Image
import cv2
import numpy as np
from scipy import stats

%matplotlib qt

In [2]:
#Use this code to see the version of python, numpy (works better on version 1.26.4 or lower)
import sys
import tensorflow as tf
import torch

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Torch:", torch.__version__)
print("SegmentEveryForam:", sef.__version__)

Python: 3.10.21 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:35:39) [MSC v.1942 64 bit (AMD64)]
NumPy: 2.2.6
TensorFlow: 2.21.0
Keras: 3.12.4
Torch: 2.13.0+cpu
SegmentEveryForam: 0.1.0


In [5]:
# ============================================================
# LOAD U-NET AND SAM 2.1 MODELS
# ============================================================

#Change the path below to point to the unet_model and sam2.1 checkpoint. 
unet_model = "../models/seg_model_foram_v2_G_ruber_30epochs.keras"
sam_checkpoint = "../models/sam2.1_hiera_large.pt"

# Load U-Net model
unet = keras.saving.load_model(unet_model, custom_objects={"weighted_crossentropy": sef.weighted_crossentropy},)

print("U-Net model loaded.")

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

# Load SAM 2.1
sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", sam_checkpoint, device=device,)
predictor = SAM2ImagePredictor(sam)

print("SAM 2.1 model loaded.")

U-Net model loaded.
Using device: cpu
SAM 2.1 model loaded.


# This is where you begin for all images

In [6]:
# ============================================================
# SELECT IMAGE(S)
# ============================================================

#The input image should not be much larger than ~2000x3000 pixels, in part to 
#avoid long running times; it is supposed to be a numpy array with 3 channels (RGB). 
#Grains should be well defined in the image and not too small (e.g., only a few pixels in size).

from tkinter import Tk
from tkinter.filedialog import askopenfilenames

root = Tk()
root.withdraw()
root.attributes("-topmost", True)

selected_files = askopenfilenames(parent=root, title="Select foram image(s)", filetypes=[("Image files", "*.jpg *.jpeg *.png *.tif *.tiff *.bmp")])

root.destroy()

image_paths = list(selected_files)

if len(image_paths) == 0:
    raise ValueError("No images were selected.")

print(f"{len(image_paths)} image(s) selected.")

for path in image_paths:
    print(path)

1 image(s) selected.
C:/Users/gabri/SegmentEveryForam/Images/Bulloides/U1559C_3H_1W_133_135_G_ruber.jpeg


In [ ]:
#Alternative way to load image, use this if the select image above does not work.
# Load your image
#image_paths = "path/to/your/image.jpg"

In [7]:
# ============================================================
# IMAGE METADATA TABLE
# Add one row per image
# ============================================================

metadata = pd.DataFrame([
    {
        "image_path": path,
        "image_id": f"img{i+1:03d}",
        "sample_id": "U1559A_3H_1W_133_135",
        "depth_ccsf": 100,
        "Age_Ma": 0,
        "species": "G_ruber"
    }
    for i, path in enumerate(image_paths)
])

metadata

,image_path,image_id,sample_id,depth_ccsf,Age_Ma,species
0,C:/Users/gabri/SegmentEveryForam/Images/Bulloi...,img001,U1559A_3H_1W_133_135,100,0,G_ruber


### For Multiple images, change the index number for each image in current_image function below

In [9]:
# ============================================================
# LOAD IMAGE AND RUN INITIAL SEGMENTATION
# ============================================================

from PIL import Image
Image.MAX_IMAGE_PIXELS = None

# Select image from metadata table
current_image = metadata.iloc[0]

fname = current_image["image_path"]
print("Processing image:")
print(fname)

# Load image
image = sfi.load_image(fname)

# Run initial SegmentEveryForam U-Net before interaction
all_grains, image_pred, all_coords = sef.predict_large_image(
    fname,
    unet,
    sam,
    use_sam=True,
    dilation=3,
    min_area=400.0,
    patch_size=2000,
    overlap=600,
    dbs_max_dist=100,
    remove_edge_grains=False
)

# Plot result
fig, ax = plt.subplots(figsize=(14, 10))

sef.plot_image_w_colorful_grains(
    image,
    all_grains,
    ax,
    cmap="tab20b",
    plot_image=True,
    im_alpha=1.0
)

ax.set_title("U-Net Segmentation before Interaction")
ax.axis("off")

plt.show()

Processing image:
C:/Users/gabri/SegmentEveryForam/Images/Bulloides/U1559C_3H_1W_133_135_G_ruber.jpeg
segmenting image tiles...


100%|██████████| 8/8 [00:07<00:00,  1.08it/s]


creating masks using SAM...


100%|██████████| 105/105 [00:19<00:00,  5.47it/s]


finding overlapping polygons...


...

57it [00:00, 11402.46it/s]


resolving 0 overlapping grain groups...


0it [00:00, ?it/s]

final grain count after merging: 57



100%|██████████| 57/57 [00:00<00:00, 135.16it/s]


In [10]:
# ============================================================
# CHECK U-NET PREDICTION AND SAM PROMPTS
# ============================================================

fig, ax = plt.subplots(figsize=(14, 10))

ax.imshow(image)
ax.imshow(image_pred.clip(0, 1), alpha=0.25)

if all_coords is not None and len(all_coords) > 0:
    ax.scatter(
        all_coords[:, 0],
        all_coords[:, 1],
        c="black",
        s=20
    )
    ax.set_title(f"U-Net prediction overlay with {len(all_coords)} SAM prompt points")
else:
    ax.set_title("U-Net prediction overlay: no SAM prompt points detected")

ax.axis("off")
plt.show()

In [12]:
# ============================================================
# EXTRACT GRAINS AND MEASURE MORPHOMETRICS
# ============================================================

# Convert segmentation polygons to Grain objects
grains = sfi.polygons_to_grains(all_grains, image=image)

# Measure each detected grain
for grain in tqdm(grains, desc="Measuring detected grains"):
    grain.measure()

print(f"\nMeasured {len(grains)} grains.")

Measuring detected grains: 100%|██████████| 57/57 [00:00<00:00, 191.92it/s]


Measured 57 grains.


#### Interacting with Segmented Output ####

In [13]:
# ============================================================
# INTERACTIVE SEGMENTATION CORRECTION
# ============================================================

# Initialize the SAM predictor with the current image
predictor.set_image(image)

# Launch the interactive editor
plot = sfi.ForamPlot(
    grains=grains,
    image=image,
    predictor=predictor,

    # Display settings
    figsize=(12, 8),
    blit=True,
    color_palette="tab20b",

    # Coloring
    color_by=None,

    # Scale information
    scale_m=500*1e-6,     # 500 microm = 0.00005 m
    px_per_m=None        # Will be determined by drawing the scale bar
)

plot.activate()

Measuring and drawing grains: 100%|██████████| 57/57 [00:00<00:00, 139.93it/s]


### Interactive controls (also shown in the figure title bar):

#### Mouse controls:

* Left-click on existing grain: Select/unselect the grain
* Left-click in grain-free area: Place foreground prompt for instant grain creation (auto-create)
#### Alt + Left-click: Place foreground prompt for multi-prompt grain creation (hold Alt, click multiple times, release Alt to create) ####
* Alt + Right-click: Place background prompt for multi-prompt creation
* Shift + Left-drag: Draw a scale bar line (red line) for unit conversion
* Middle-click or Shift + Left-click on grain: Show grain measurement info

#### Keyboard controls:

* d or Delete: Delete selected (highlighted) object
* a/m: add/merge selected foram segments (selected segments must form one connected object)
* z: Undo (delete the most recently created segment/object)
* Ctrl (hold): Temporarily hide all segment masks
* Esc: Remove all prompts and unselect all forams
* c: Create foram segments from existing prompts (alternative to auto-create)

After editing, retrieve the updated grains and deactivate the interactive features:

## Confirm that you have selected all specimens and drawn scale before progressing

In [14]:
# ============================================================
# RETRIEVE CORRECTED GRAINS FROM INTERACTIVE PLOT
# ============================================================

grains = plot.get_grains()

print(f"Retrieved {len(grains)} corrected grains from ForamPlot.")

Retrieved 54 corrected grains from ForamPlot.


In [15]:
# Turn off interactive features
plot.deactivate()

# Draw the major and minor axes of each grain
plot.draw_axes()

### Saving the corrected mask for further fine tuning

In [146]:
# ============================================================
# SAVE FINAL CORRECTED MASK
# ============================================================

from pathlib import Path

# Extract final polygons from corrected grains
polygons = [grain.polygon for grain in grains]

# Create labeled image and binary/all-grain mask
labels, mask_all = sef.create_labeled_image(polygons, image)

# Create output folder
output_dir = Path("mask_outputs")
output_dir.mkdir(exist_ok=True)

sample_id = current_image["sample_id"]

mask_path = output_dir / f"{sample_id}_{current_image['species']}_mask.png"

# Save mask
cv2.imwrite(str(mask_path), mask_all.astype(np.uint8))

print(f"Saved corrected mask to: {mask_path}")

Saved corrected mask to: mask_outputs\U1559A_1H_1W_92_94_G_bulloides_mask.png


### To get the pixels per meters

In [12]:
# ============================================================
# SUMMARIZE GRAIN MORPHOMETRICS
# ============================================================

# Pixel-to-meter conversion from the interactive scale bar
px_per_m = plot.px_per_m
if px_per_m is None:
    raise ValueError("No scale bar found. Draw the scale bar before running morphometrics.")

print(f"Calibration: {px_per_m:.2f} pixels per meter")

# Compute morphometric summary
summary = sfi.get_summary(grains, px_per_m)

Calibration: 529374.94 pixels per meter


### For repeated segmentation, run cell below to skip measuring scale bar if scale stays the same

In [645]:
# ============================================================
# PIXEL CALIBRATION
# ============================================================

PX_PER_M = 228220.37/526894.56/862647.30/360915.04/363020.85/532133.57

summary = sfi.get_summary(grains, PX_PER_M)

In [111]:
# ============================================================
# SPECIMEN-LEVEL FORAM MORPHOMETRICS
# Units: mm and mm²
# ============================================================

# Make foram_data have the same rows/index as summary
foram_data = pd.DataFrame(index=summary.index)

# Metadata repeated for every specimen
foram_data["sample_id"] = current_image["sample_id"]
foram_data["image_id"] = current_image["image_id"]
foram_data["species"] = current_image["species"]
foram_data["depth_ccsf"] = current_image["depth_ccsf"]
foram_data["Age_Ma"] = current_image["Age_Ma"]

# Grain ID
foram_data["grain_id"] = np.arange(1, len(summary) + 1)

# Core morphometrics
foram_data["area_mm2"] = summary["area"] * 1_000_000
foram_data["perimeter_mm"] = summary["perimeter"] * 1000
foram_data["major_axis_mm"] = summary["major_axis_length"] * 1000
foram_data["minor_axis_mm"] = summary["minor_axis_length"] * 1000
foram_data["orientation_rad"] = summary["orientation"]

#derived metrics
# Shape ratios
foram_data["aspect_ratio"] = foram_data["major_axis_mm"] / foram_data["minor_axis_mm"]
foram_data["elongation"] = 1 - (foram_data["minor_axis_mm"] / foram_data["major_axis_mm"])
foram_data["roundness"] = foram_data["minor_axis_mm"] / foram_data["major_axis_mm"]

# Area-based diameter
foram_data["equivalent_diameter_mm"] = np.sqrt(
    4 * foram_data["area_mm2"] / np.pi
)

# Compactness / circularity
foram_data["circularity"] = (
    4 * np.pi * foram_data["area_mm2"] / (foram_data["perimeter_mm"] ** 2)
)

# Perimeter-based shape complexity
foram_data["perimeter_area_ratio"] = (
    foram_data["perimeter_mm"] / foram_data["area_mm2"]
)

# Orientation in degrees
foram_data["orientation_deg"] = np.degrees(foram_data["orientation_rad"])

# Location in image, still in pixels
foram_data["centroid_y_px"] = summary["centroid-0"]
foram_data["centroid_x_px"] = summary["centroid-1"]

foram_data.head()

,sample_id,image_id,species,depth_ccsf,Age_Ma,grain_id,area_mm2,perimeter_mm,major_axis_mm,minor_axis_mm,orientation_rad,aspect_ratio,elongation,roundness,equivalent_diameter_mm,circularity,perimeter_area_ratio,orientation_deg,centroid_y_px,centroid_x_px
0,401_U1611A_63R_CC_0_5,img001,Orbulina,1150.765,0,1,0.093753,1.146902,0.349101,0.342041,-1.197948,1.020641,0.020224,0.979776,0.345500,0.895661,12.233185,-68.637370,218.645005,898.903461
1,401_U1611A_63R_CC_0_5,img001,Orbulina,1150.765,0,2,0.137739,1.387869,0.424587,0.413167,-0.852215,1.027640,0.026896,0.973104,0.418777,0.898604,10.076109,-48.828321,342.663959,697.168981
2,401_U1611A_63R_CC_0_5,img001,Orbulina,1150.765,0,3,0.090336,1.132211,0.342541,0.336135,-0.669424,1.019058,0.018701,0.981299,0.339146,0.885560,12.533277,-38.355175,380.848537,1247.355962
3,401_U1611A_63R_CC_0_5,img001,Orbulina,1150.765,0,4,0.098791,1.195836,0.362990,0.347521,-0.153007,1.044511,0.042614,0.957386,0.354661,0.868129,12.104707,-8.766635,530.763715,995.814901
4,401_U1611A_63R_CC_0_5,img001,Orbulina,1150.765,0,5,0.139772,1.407305,0.423135,0.420727,0.735359,1.005723,0.005691,0.994309,0.421857,0.886857,10.068573,42.132982,702.106217,680.131694


### To display final grains with corresponding grain_id

In [82]:
# ============================================================
# DISPLAY FINAL GRAINS WITH GRAIN IDs
# ============================================================

from shapely.geometry import Polygon

fig, ax = plt.subplots(figsize=(15, 10))

# Show original image
ax.imshow(image)

for grain, grain_id in zip(grains, foram_data["grain_id"]):

    # Polygon coordinates
    polygon = Polygon(grain.polygon)

    # Centroid
    centroid = polygon.centroid

    # Display grain_id
    ax.text(
        centroid.x,
        centroid.y,
        str(grain_id),
        color="yellow",
        fontsize=8,
        ha="center",
        va="center",
        bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
    )

ax.set_title("Final segmented foraminifera with grain IDs")
ax.axis("off")

plt.show()

#### Save the foram data

In [112]:
# ============================================================
# SAVE / APPEND SPECIMEN-LEVEL FORAM DATA
# ============================================================

from pathlib import Path

output_dir = Path("foram_morphometrics_outputs")
output_dir.mkdir(exist_ok=True)

# One spreadsheet per sample/depth/species
sample_id = current_image["sample_id"]
species = current_image["species"]

foram_data_path = output_dir / f"{sample_id}_{species}_foram_data.csv"

# If file already exists, append current image data
if foram_data_path.exists():
    existing_data = pd.read_csv(foram_data_path)
    combined_data = pd.concat([existing_data, foram_data], ignore_index=True)
else:
    combined_data = foram_data.copy()

# Remove duplicate specimens (e.g., if the same image is processed twice)
combined_data = combined_data.drop_duplicates(
    subset=["image_id", "grain_id"],
    keep="first"
)

# Save updated spreadsheet
combined_data.to_csv(foram_data_path, index=False)

print(f"Saved foram data to: {foram_data_path}")
print(f"Total rows in file: {len(combined_data)}")

Saved foram data to: foram_morphometrics_outputs\401_U1611A_63R_CC_0_5_Orbulina_foram_data.csv
Total rows in file: 23


### Create average measurement of the foram data 

In [113]:
# ============================================================
# SAVE / APPEND SAMPLE-LEVEL MORPHOMETRIC SUMMARY
# ============================================================

# Read the full foram data file for this sample/species
combined_data = pd.read_csv(foram_data_path)

# Compute one summary row for this sample/species
sample_summary = pd.DataFrame([{
    "sample_id": current_image["sample_id"],
    "species": current_image["species"],
    "depth_ccsf": current_image["depth_ccsf"],
    "Age_Ma": current_image["Age_Ma"],
    "n_specimens": len(combined_data),
    "image_ids": ",".join(sorted(combined_data["image_id"].unique())),

    "mean_area_mm2": combined_data["area_mm2"].mean(),
    "median_area_mm2": combined_data["area_mm2"].median(),
    "std_area_mm2": combined_data["area_mm2"].std(),
    "p95_area_mm2": combined_data["area_mm2"].quantile(0.95),

    "mean_perimeter_mm": combined_data["perimeter_mm"].mean(),
    "median_perimeter_mm": combined_data["perimeter_mm"].median(),
    "std_perimeter_mm": combined_data["perimeter_mm"].std(),
    "p95_perimeter_mm": combined_data["perimeter_mm"].quantile(0.95),


    "mean_major_axis_mm": combined_data["major_axis_mm"].mean(),
    "median_major_axis_mm": combined_data["major_axis_mm"].median(),
    "std_major_axis_mm": combined_data["major_axis_mm"].std(),
    "p95_major_axis_mm": combined_data["major_axis_mm"].quantile(0.95),


    "mean_minor_axis_mm": combined_data["minor_axis_mm"].mean(),
    "median_minor_axis_mm": combined_data["minor_axis_mm"].median(),
    "std_minor_axis_mm": combined_data["minor_axis_mm"].std(),
    "p95_minor_axis_mm": combined_data["minor_axis_mm"].quantile(0.95),

    "mean_equivalent_diameter_mm": combined_data["equivalent_diameter_mm"].mean(),
    "median_equivalent_diameter_mm": combined_data["equivalent_diameter_mm"].median(),
    "std_equivalent_diameter_mm": combined_data["equivalent_diameter_mm"].std(),
    "p95_equivalent_diameter_mm": combined_data["equivalent_diameter_mm"].quantile(0.95),

    "mean_aspect_ratio": combined_data["aspect_ratio"].mean(),
    "median_aspect_ratio": combined_data["aspect_ratio"].median(),
    "std_aspect_ratio": combined_data["aspect_ratio"].std(),

    "mean_circularity": combined_data["circularity"].mean(),
    "median_circularity": combined_data["circularity"].median(),
    "std_circularity": combined_data["circularity"].std(),

    "mean_elongation": combined_data["elongation"].mean(),
    "mean_roundness": combined_data["roundness"].mean(),
    "mean_perimeter_area_ratio": combined_data["perimeter_area_ratio"].mean()
}])

In [114]:
from pathlib import Path

summary_output_dir = Path("foram_average_measurement_outputs")
summary_output_dir.mkdir(exist_ok=True)

# One summary spreadsheet per species
species = current_image["species"]
sample_summary_path = (summary_output_dir / f"{species}_sample_summary.csv")
# Append to existing species summary file
if sample_summary_path.exists():
    existing_summary = pd.read_csv(sample_summary_path)
    all_sample_summary = pd.concat([existing_summary, sample_summary], ignore_index=True)
else:
    all_sample_summary = sample_summary.copy()

# Remove duplicate sample/species/depth rows if rerun
all_sample_summary = all_sample_summary.drop_duplicates(
    subset=["sample_id", "species", "depth_ccsf"],
    keep="last"
)

# Sort by depth
all_sample_summary = all_sample_summary.sort_values("depth_ccsf").reset_index(drop=True)

# Save
all_sample_summary.to_csv(sample_summary_path, index=False)

print(f"Saved sample summary to: {sample_summary_path}")
display(sample_summary)

Saved sample summary to: foram_average_measurement_outputs\Orbulina_sample_summary.csv


,sample_id,species,depth_ccsf,Age_Ma,n_specimens,image_ids,mean_area_mm2,median_area_mm2,std_area_mm2,p95_area_mm2,...,p95_equivalent_diameter_mm,mean_aspect_ratio,median_aspect_ratio,std_aspect_ratio,mean_circularity,median_circularity,std_circularity,mean_elongation,mean_roundness,mean_perimeter_area_ratio
0,401_U1611A_63R_CC_0_5,Orbulina,1150.765,0,23,img001,0.111505,0.11479,0.038944,0.171654,...,0.467365,1.033004,1.02764,0.01947,0.888396,0.889589,0.01202,0.031626,0.968374,11.918905


## Can Stop here if you don't want to see plots

In [29]:
# ============================================================
# HISTOGRAM OF SPECIMEN SIZE
# ============================================================

figure_output_dir = Path("foram_figure_outputs")
figure_output_dir.mkdir(exist_ok=True)

sample_id = current_image["sample_id"]
species = current_image["species"]
image_id = current_image["image_id"]

fig, ax = plt.subplots(figsize=(8, 6))

ax.hist(
    foram_data["major_axis_mm"],
    bins=20,
    edgecolor="black"
)

ax.set_xlabel("Major axis length (mm)")
ax.set_ylabel("Number of specimens")
ax.set_title(f"{sample_id} | {species} | {image_id}\nSpecimen size distribution")

plt.tight_layout()

hist_path = figure_output_dir / f"{sample_id}_{species}_{image_id}_major_axis_histogram.png"
fig.savefig(hist_path, dpi=300, bbox_inches="tight")

plt.show()

print(f"Saved histogram to: {hist_path}")

Saved histogram to: foram_figure_outputs\401_U1611A_50R_1W_91_95_Orbulina_img001_major_axis_histogram.png


## To read all foram_data and generate a box plot by depth

In [130]:
# ============================================================
# BOXPLOT OF SPECIMEN SIZE BY DEPTH
# Reads all CSVs for one species from foram_morphometrics_outputs
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

morph_dir = Path("foram_morphometrics_outputs")
figure_output_dir = Path("foram_figure_outputs")
figure_output_dir.mkdir(exist_ok=True)

species = current_image["species"]

csv_files = list(morph_dir.glob(f"*_{species}_foram_data.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No foram_data CSV files found for species: {species}"
    )

all_foram_data = pd.concat(
    [pd.read_csv(file) for file in csv_files],
    ignore_index=True
)

all_foram_data = all_foram_data.sort_values("depth_ccsf")

depths = sorted(all_foram_data["depth_ccsf"].dropna().unique())

boxplot_data = [
    all_foram_data.loc[
        all_foram_data["depth_ccsf"] == depth,
        "major_axis_mm"
    ].dropna()
    for depth in depths
]

# Number of specimens at each depth
depth_summary = (
    all_foram_data
    .groupby("depth_ccsf")
    .size()
    .reset_index(name="n_specimens")
)

# Appearance
meanprops = dict(
    marker='o',
    markerfacecolor='red',
    markeredgecolor='black',
    markersize=6
)

boxprops = dict(
    facecolor='lightblue',
    edgecolor='black'
)

medianprops = dict(
    color='orange',
    linewidth=2
)

fig, ax = plt.subplots(figsize=(8, 10))

ax.boxplot(
    boxplot_data,
    positions=depths,
    vert=False,
    widths=0.6,
    notch=True,
    showmeans=True,
    patch_artist=True,
    meanprops=meanprops,
    boxprops=boxprops,
    medianprops=medianprops
)


# Depth labels with specimen count
ytick_labels = [
    f"{row.depth_ccsf:.2f} (n={int(row.n_specimens)})"
    for _, row in depth_summary.iterrows()
]

ax.set_yticks(depth_summary["depth_ccsf"])
ax.set_yticklabels(ytick_labels)

ax.set_xlabel("Major axis length (mm)")
ax.set_ylabel("Depth (ccsf)")
ax.set_title(f"{species}: Major axis length distribution by depth")

ax.invert_yaxis()

plt.tight_layout()

boxplot_depth_path = (
    figure_output_dir /
    f"{species}_major_axis_by_depth_boxplot.png"
)

fig.savefig(
    boxplot_depth_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Read {len(csv_files)} files:")
for file in csv_files:
    print("•", file.name)

print(f"Saved figure to: {boxplot_depth_path}")

Read 7 files:
• 401_U1611A_48R_1W_122_126_Orbulina_foram_data.csv
• 401_U1611A_50R_1W_91_95_Orbulina_foram_data.csv
• 401_U1611A_51R_CC_14_19_Orbulina_foram_data.csv
• 401_U1611A_52R_4W_26_30_Orbulina_foram_data.csv
• 401_U1611A_54R_2W_31_35_Orbulina_foram_data.csv
• 401_U1611A_56R_2W_9_13_Orbulina_foram_data.csv
• 401_U1611A_63R_CC_0_5_Orbulina_foram_data.csv
Saved figure to: foram_figure_outputs\Orbulina_major_axis_by_depth_boxplot.png


## plot by depth with connecting line

In [137]:
# ============================================================
# BOXPLOT OF SPECIMEN SIZE BY DEPTH
# Reads all CSVs for one species from foram_morphometrics_outputs
# Includes a connecting line through mean specimen size
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# DIRECTORIES
# ------------------------------------------------------------

morph_dir = Path("foram_morphometrics_outputs")
figure_output_dir = Path("foram_figure_outputs")
figure_output_dir.mkdir(exist_ok=True)

# Current species
species = current_image["species"]

# ------------------------------------------------------------
# FIND ALL MORPHOMETRIC CSV FILES FOR THIS SPECIES
# ------------------------------------------------------------

csv_files = list(
    morph_dir.glob(f"*_{species}_foram_data.csv")
)

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No foram_data CSV files found for species: {species}"
    )

# ------------------------------------------------------------
# COMBINE ALL FILES
# ------------------------------------------------------------

all_foram_data = pd.concat(
    [pd.read_csv(file) for file in csv_files],
    ignore_index=True
)

# Sort by depth
all_foram_data = all_foram_data.sort_values("depth_ccsf")

# ------------------------------------------------------------
# GET UNIQUE DEPTHS
# ------------------------------------------------------------

depths = sorted(
    all_foram_data["depth_ccsf"]
    .dropna()
    .unique()
)

# ------------------------------------------------------------
# PREPARE DATA FOR BOXPLOTS
# ------------------------------------------------------------

boxplot_data = [
    all_foram_data.loc[
        all_foram_data["depth_ccsf"] == depth,
        "major_axis_mm"
    ].dropna()
    for depth in depths
]

# ------------------------------------------------------------
# NUMBER OF SPECIMENS AT EACH DEPTH
# ------------------------------------------------------------

depth_summary = (
    all_foram_data
    .groupby("depth_ccsf")
    .size()
    .reindex(depths)
    .reset_index(name="n_specimens")
)

# ------------------------------------------------------------
# CALCULATE MEAN SIZE AT EACH DEPTH
# ------------------------------------------------------------

mean_sizes = (
    all_foram_data
    .groupby("depth_ccsf")["major_axis_mm"]
    .mean()
    .reindex(depths)
)

# ------------------------------------------------------------
# BOXPLOT APPEARANCE
# ------------------------------------------------------------

meanprops = dict(
    marker="o",
    markerfacecolor="red",
    markeredgecolor="black",
    markersize=6
)

boxprops = dict(
    facecolor="lightblue",
    edgecolor="black"
)

medianprops = dict(
    color="orange",
    linewidth=2
)

whiskerprops = dict(
    color="black"
)

capprops = dict(
    color="black"
)

# ------------------------------------------------------------
# CREATE FIGURE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 10))

# ------------------------------------------------------------
# PLOT BOXPLOTS
# ------------------------------------------------------------

ax.boxplot(
    boxplot_data,
    positions=depths,
    vert=False,
    widths=0.6,
    notch=True,
    showmeans=True,
    patch_artist=True,
    meanprops=meanprops,
    boxprops=boxprops,
    medianprops=medianprops,
    whiskerprops=whiskerprops,
    capprops=capprops
)

# ------------------------------------------------------------
# CONNECT MEAN VALUES ACROSS DEPTHS
# ------------------------------------------------------------

ax.plot(
    mean_sizes.values,
    depths,
    color="red",
    linewidth=1.5,
    marker="o",
    markersize=6,
    markerfacecolor="red",
    markeredgecolor="black",
    zorder=4,
    label="Mean specimen size"
)
ax.legend(loc="upper left")

# ------------------------------------------------------------
# DEPTH LABELS WITH SPECIMEN COUNT
# ------------------------------------------------------------

ytick_labels = [
    f"{depth:.2f} (n={int(n)})"
    for depth, n in zip(
        depth_summary["depth_ccsf"],
        depth_summary["n_specimens"]
    )
]

ax.set_yticks(depths)
ax.set_yticklabels(ytick_labels)

# ------------------------------------------------------------
# AXIS LABELS AND TITLE
# ------------------------------------------------------------

ax.set_xlabel("Major axis length (mm)")
ax.set_ylabel("Depth (ccsf)")
ax.set_title(
    f"{species}: Major axis length distribution by depth"
)

# Depth increases downward
ax.invert_yaxis()

# Legend for connecting mean line
ax.legend()

# ------------------------------------------------------------
# FINALIZE FIGURE
# ------------------------------------------------------------

plt.tight_layout()

# ------------------------------------------------------------
# SAVE FIGURE
# ------------------------------------------------------------

boxplot_depth_path = (
    figure_output_dir /
    f"{species}_major_axis_by_depth_boxplot.png"
)

fig.savefig(
    boxplot_depth_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------

print(f"Read {len(csv_files)} files:")

for file in csv_files:
    print("•", file.name)

print()
print(f"Saved figure to: {boxplot_depth_path}")

Read 7 files:
• 401_U1611A_48R_1W_122_126_Orbulina_foram_data.csv
• 401_U1611A_50R_1W_91_95_Orbulina_foram_data.csv
• 401_U1611A_51R_CC_14_19_Orbulina_foram_data.csv
• 401_U1611A_52R_4W_26_30_Orbulina_foram_data.csv
• 401_U1611A_54R_2W_31_35_Orbulina_foram_data.csv
• 401_U1611A_56R_2W_9_13_Orbulina_foram_data.csv
• 401_U1611A_63R_CC_0_5_Orbulina_foram_data.csv

Saved figure to: foram_figure_outputs\Orbulina_major_axis_by_depth_boxplot.png


### Plot 95th percentile vs depth

In [120]:
# ============================================================
# DOWNCORE 95TH PERCENTILE OF MAJOR AXIS LENGTH
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

morph_dir = Path("foram_morphometrics_outputs")
figure_output_dir = Path("foram_figure_outputs")
figure_output_dir.mkdir(exist_ok=True)

species = current_image["species"]

csv_files = list(morph_dir.glob(f"*_{species}_foram_data.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No foram_data CSV files found for species: {species}"
    )

all_foram_data = pd.concat(
    [pd.read_csv(file) for file in csv_files],
    ignore_index=True
)

# Compute statistics by depth
depth_summary = (
    all_foram_data
    .groupby("depth_ccsf")["major_axis_mm"]
    .agg(
        n_specimens="count",
        p95_major_axis_mm=lambda x: x.quantile(0.95)
    )
    .reset_index()
    .sort_values("depth_ccsf")
)

fig, ax = plt.subplots(figsize=(6, 8))

# 95th percentile profile
ax.plot(
    depth_summary["p95_major_axis_mm"],
    depth_summary["depth_ccsf"],
    "-o",
    linewidth=2,
    markersize=6
)

# Annotate specimen count
for _, row in depth_summary.iterrows():
    ax.text(
        row["p95_major_axis_mm"] + 0.003,
        row["depth_ccsf"],
        f"n={int(row['n_specimens'])}",
        fontsize=8,
        va="center"
    )

ax.set_xlabel("95th percentile major axis (mm)")
ax.set_ylabel("Depth (ccsf)")
ax.set_title(f"{species}: 95th percentile major axis")

ax.invert_yaxis()

plt.tight_layout()

figure_path = (
    figure_output_dir /
    f"{species}_p95_major_axis_downcore.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Read {len(csv_files)} files:")
for file in csv_files:
    print("•", file.name)

print(f"Saved figure to: {figure_path}")

Read 7 files:
• 401_U1611A_48R_1W_122_126_Orbulina_foram_data.csv
• 401_U1611A_50R_1W_91_95_Orbulina_foram_data.csv
• 401_U1611A_51R_CC_14_19_Orbulina_foram_data.csv
• 401_U1611A_52R_4W_26_30_Orbulina_foram_data.csv
• 401_U1611A_54R_2W_31_35_Orbulina_foram_data.csv
• 401_U1611A_56R_2W_9_13_Orbulina_foram_data.csv
• 401_U1611A_63R_CC_0_5_Orbulina_foram_data.csv
Saved figure to: foram_figure_outputs\Orbulina_p95_major_axis_downcore.png


### To generate scatter plot by depth using sample_summary

In [119]:
# ============================================================
# DOWNCORE MEAN MORPHOMETRIC PROFILE
# WITH STANDARD ERROR OF THE MEAN (SEM)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

summary_dir = Path("foram_average_measurement_outputs")
figure_output_dir = Path("foram_figure_outputs")
figure_output_dir.mkdir(exist_ok=True)

species = current_image["species"]

summary_path = summary_dir / f"{species}_sample_summary.csv"

summary = pd.read_csv(summary_path)
summary = summary.sort_values("depth_ccsf")

# Calculate Standard Error of the Mean (SEM)
summary["sem_major_axis_mm"] = (
    summary["std_major_axis_mm"] /
    np.sqrt(summary["n_specimens"])
)

fig, ax = plt.subplots(figsize=(6, 8))

# Mean ± SEM
ax.errorbar(
    summary["mean_major_axis_mm"],
    summary["depth_ccsf"],
    xerr=summary["sem_major_axis_mm"],
    fmt='none',              # <-- only error bars
    ecolor='tab:blue',
    elinewidth=1.5,
    capsize=4,
    capthick=1,
    zorder=1
)

# ============================================================
# MEAN PROFILE (ORANGE)
# ============================================================

ax.plot(
    summary["mean_major_axis_mm"],
    summary["depth_ccsf"],
    'o-',
    color='tab:orange',
    linewidth=2,
    markersize=6,
    zorder=2
)
# Annotate specimen count
for _, row in summary.iterrows():
    ax.text(
        row["mean_major_axis_mm"] + row["sem_major_axis_mm"] + 0.003,
        row["depth_ccsf"],
        f"n={int(row['n_specimens'])}",
        fontsize=8,
        va="center"
    )

ax.set_xlabel("Mean major axis (mm)")
ax.set_ylabel("Depth (ccsf)")
ax.set_title(f"{species}: Mean major axis ± SEM")

# Geological convention
ax.invert_yaxis()

plt.tight_layout()

figure_path = (
    figure_output_dir /
    f"{species}_mean_major_axis_downcore_SEM.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Saved figure to: {figure_path}")

Saved figure to: foram_figure_outputs\Orbulina_mean_major_axis_downcore_SEM.png
